# VisionServeX — Colab GPU Worker

Run VisionServeX on a Google Colab GPU as a **temporary** remote worker.

**Not for production.** Colab sessions can disconnect at any time. Use this for demos, benchmarks, and short-lived tasks only.

**Before you start:** in the menu, choose **Runtime → Change runtime type → GPU**.

## 1. Install VisionServeX

In [ ]:
!pip install -q -U 'visionservex[server,hf,rfdetr]'
!visionservex version

## 2. Diagnose the Colab environment

In [ ]:
!visionservex colab doctor

## 3. (Optional) Persistent cache via Google Drive

Without Drive, model downloads are lost when the Colab session ends. Mount Drive and use a persistent cache directory:

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%env VISIONSERVEX_CACHE_DIR=/content/drive/MyDrive/visionservex_cache
!mkdir -p $VISIONSERVEX_CACHE_DIR
!visionservex colab cache-path

## 4. Pull a GPU-friendly model suite

In [ ]:
!visionservex pull-suite gpu-demo --yes

## 5. Start the gateway with the `colab-gpu-worker` profile

Conservative defaults: single model loaded, single concurrency, 1.5 GB free VRAM kept. Local-only bind.

In [ ]:
import subprocess, time
proc = subprocess.Popen(['visionservex', 'gateway', 'start', '--profile', 'colab-gpu-worker'])
time.sleep(4)
!curl -s http://127.0.0.1:8080/health || echo 'gateway not ready yet'

## 6. Run an inference from Python

In [ ]:
from visionservex import Client
from PIL import Image
import io, requests

# fetch a small sample image
url = 'https://raw.githubusercontent.com/arashsajjadi/VisionServeX/main/examples/images/street.jpg'
r = requests.get(url, timeout=10)
with open('/content/street.jpg', 'wb') as f:
    f.write(r.content)

client = Client('http://127.0.0.1:8080')
result = client.detect('dfine-n', '/content/street.jpg')
print(result.summary())

## 7. (Optional) Expose via Cloudflare Tunnel — auth required

**Read first:** [docs/cloudflare_tunnel.md](https://github.com/arashsajjadi/VisionServeX/blob/main/docs/cloudflare_tunnel.md) and [docs/security.md](https://github.com/arashsajjadi/VisionServeX/blob/main/docs/security.md).

The CLI refuses to start a tunnel without (a) an API key configured and (b) `--i-understand-this-is-public`.

In [ ]:
# Generate a one-time API token (shown ONCE)
!visionservex colab token

In [ ]:
# Paste the token from the previous cell and enable auth:
%env VISIONSERVEX_AUTH__API_KEY=<paste-token>
%env VISIONSERVEX_AUTH__ENABLED=true

# Install cloudflared, then start tunnel
# !apt-get install -y cloudflared || pip install cloudflared
# !visionservex colab tunnel-start --domain your-domain.example.com --i-understand-this-is-public

## 8. Cleanup before disconnecting

In [ ]:
!visionservex colab tunnel-stop || true
!visionservex colab cleanup --yes || true

---

## Known limitations

- Sessions can disconnect without warning.
- GPU model varies (T4, L4, A100). VisionServeX adapts via the VRAM guard.
- Without Drive, model downloads are lost between sessions.
- Public exposure requires auth + explicit acknowledgement — the CLI refuses otherwise.
- Do not store secrets or sensitive data in notebook cells.